In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

In [ ]:
train.head()

In [ ]:
train.dtypes.value_counts()

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

In [ ]:
garage_cols = ['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond']

# Check if the NaN values are in the same rows
garage_nan = train[garage_cols].isnull()
print("Rows where ALL 5 columns in the Garage table are NaN:")
print((garage_nan.sum(axis=1) == 5).sum())

print("\nRows where ANY column in Garage is NaN:")
print((garage_nan.sum(axis=1) > 0).sum())

In [ ]:
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
bsmt_nan = train[bsmt_cols].isnull()

# Rows with partial NaN (1-4 missing, not all 5)
partial_nan_mask = (bsmt_nan.sum(axis=1) > 0) & (bsmt_nan.sum(axis=1) < 5)
problem_rows = train[partial_nan_mask]

# Show problematic rows + TotalBsmtSF to check if basement physically exists
print(problem_rows[bsmt_cols + ['TotalBsmtSF']])

### Basement columns — anomaly investigation

Unlike Garage (81 rows, all 5 cols NaN = no garage), Bsmt has:
- **37 rows**: all 5 cols NaN → no basement (impute 'None')
- **2 rows**: partial NaN → basement exists, data entry error

Strategy:
- Row 332 (`BsmtFinType2` missing, TotalBsmtSF=3206): impute with mode
- Row 948 (`BsmtExposure` missing, TotalBsmtSF=936): impute with mode
- Remaining 37 rows: impute 'None' (no basement)